# Neural Adaptation — Afferent-Gain Example Plots

Two-mechanism model (built on `figure_example_plots.ipynb`, inspired by the
input-depression idea in `test_model_adj_1.ipynb`, but **not** an identity counter):

1. **Recurrent Hebbian facilitation on the outputs** — a multiplicative state
   `eff` on `W`, driven by `outer(r, r)` (the response, not the input),
   recovering toward 1 with `TAU_HEBBIAN`. `ETA_HEBBIAN > 0` (facilitation).

2. **Afferent (feedforward) gain plasticity** — a per-input-synapse gain vector
   `g` (length N). Each synapse is facilitated/depressed by the **raw input
   magnitude `I_raw` it receives**, recovering toward 1 with `TAU_AFFERENT`.
   Effective drive `I = g ⊙ I_raw`. `ETA_AFFERENT < 0` (depression).
   Stimulus-dependence emerges from the magnitude pattern — no per-stimulus
   bookkeeping.

Inputs are **sparse-uniform** by default (each stimulus activates a distinct
random `SPARSITY·N` subset with magnitudes `~U[0,1]`; switch `INPUT_MODE` to
`dense_uniform` for the all-neuron variant). The recurrent matrix is
parametrised in **dynamics units**: entries `~ Normal(W_MEAN/N, W_SD/√N)`, so
`W_MEAN` is the common-mode eigenvalue and `W_SD` the random spectral radius
(both N-independent, no rescaling needed). When `NORMALIZE_EV` is on, each step
resets the random part to radius `W_SD` while pinning the mean to `W_MEAN`, so
Hebbian plasticity reshapes connectivity structure at fixed mean and radius.

**Outputs** (PNG + EPS to `./output_examples_afferent/{param_tag}/`):
- the full standard suite per network and averaged (`netavg_`): entropy-binned
  activity heatmaps, stimulus correlation matrices, 3 PCA colourings × 2 rows,
  PCA trajectory, PCA with entropy/activity axes;
- plus the **boring** vs **max-entropy** 2×5 mean-activity panels per network.


In [ ]:
# ======================
# Neural Adaptation — Afferent-Gain Example Visualization Script
# ======================
# Two mechanisms only (both "Hebb-like"):
#   - recurrent Hebbian facilitation on the OUTPUTS r  (eff on W)
#   - afferent feedforward gain plasticity driven by the RAW input magnitude (g)
# Set everything in the USER SETTINGS block below.
# ======================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
import os

# ================================================================
# >>>  USER SETTINGS
# ================================================================
# --- recurrent weight matrix, parametrised in DYNAMICS units ---
#   entries ~ Normal(W_MEAN / N, W_SD / sqrt(N))  so that:
#     W_MEAN = common-mode (all-ones) eigenvalue   [entry-mean * N]
#     W_SD   = random-part spectral radius         [entry-SD  * sqrt(N)]
#   Both are N-independent; keep each < 1 for stability.
W_MEAN          =  0.25
W_SD            =  0.9

# --- the two adaptation rates ---
ETA_HEBBIAN     =  1     # recurrent Hebbian on outputs r   ( >0 = facilitation )
ETA_AFFERENT    = -0.2     # afferent gain on raw input I_raw  ( <0 = depression  )

# --- two separate recovery time constants (in sequence steps) ---
TAU_HEBBIAN     =  10.0    # recovery of recurrent eff  -> 1
TAU_AFFERENT    =  10.0    # recovery of afferent gain  -> 1

# --- optional: renormalise W_eff each step (pins mean & random radius) ---
NORMALIZE_EV    =  True   # OFF by default; flip to True to test

# --- afferent gain clip bounds (wide, so strong etas do not saturate) ---
MIN_GAIN        = -10.0
MAX_GAIN        =  10.0

# --- input structure ---
INPUT_MODE      = 'sparse_uniform'   # 'sparse_uniform' | 'dense_uniform'
SPARSITY        =  0.1               # fraction of neurons active per stimulus (sparse mode)

N_NETWORKS      =  50
# ================================================================

# ================================================================
# Font sizes
# ================================================================
FONT_SIZE_BASE  = 22
FONT_SIZE_TITLE = 26
FONT_SIZE_TICK  = 20
FONT_SIZE_CBAR  = 22

plt.rcParams.update({
    'font.size':        FONT_SIZE_BASE,
    'axes.titlesize':   FONT_SIZE_TITLE,
    'axes.labelsize':   FONT_SIZE_BASE,
    'xtick.labelsize':  FONT_SIZE_TICK,
    'ytick.labelsize':  FONT_SIZE_TICK,
})

np.random.seed(42)

# ================================================================
# Simulation parameters
# ================================================================
N             = 100
stim_strength = 1.0
theta0        = 0.0
tau           = 1.0
dt            = 0.01
n_iter        = 1000
N_BINS        = 5

ALL_STIMULI   = [chr(ord('A') + i) for i in range(10)]
NUM_TO_LETTER = {i + 1: chr(ord('A') + i) for i in range(10)}
STIM_TO_IDX   = {s: i for i, s in enumerate(ALL_STIMULI)}

# Stimulus colormap as proper LUT
STIM_CMAP   = plt.cm.tab10
STIM_NORM   = mcolors.BoundaryNorm(np.arange(-0.5, 10.5, 1), STIM_CMAP.N)
STIM_COLORS = [STIM_CMAP(STIM_NORM(i)) for i in range(10)]

# ================================================================
# Parameter tag & output directory
# ================================================================
def _fmt(v):
    sign = 'm' if v < 0 else ''
    av   = abs(v)
    ip   = int(av)
    dp   = round((av - ip) * 100)
    return f'{sign}{ip:04d}_{dp:02d}'

PARAM_TAG = (f'wm{_fmt(W_MEAN)}_sd{_fmt(W_SD)}'
             f'_heb{_fmt(ETA_HEBBIAN)}_aff{_fmt(ETA_AFFERENT)}'
             f'_th{_fmt(TAU_HEBBIAN)}_ta{_fmt(TAU_AFFERENT)}'
             f'_ne{int(NORMALIZE_EV)}_{INPUT_MODE[:2]}{int(round(SPARSITY*100))}')
OUTPUT_DIR = os.path.join('./output_examples_afferent', PARAM_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# VAL DATA
# ================================================================
VAL_DATA = np.array([
    [[2,7,7,7,2,2,6,7,7,7, 8, 2, 7, 1, 2, 1, 6, 8, 9, 4],
     [2,7,2,7,7,2,2,7,6,2, 6, 5, 8, 5, 6, 6, 5, 8, 5, 7],
     [2,7,7,7,2,2,7,2,8,2, 2, 2, 5, 8, 1, 6, 6, 8, 3, 3],
     [2,2,7,7,7,2,6,2,8,2, 7, 6, 8, 7, 8, 7, 6, 2, 2,10],
     [2,2,2,7,2,2,7,7,6,8, 2, 5, 6, 5, 6, 7, 6, 8, 6, 8],
     [2,2,2,7,2,7,2,7,7,8, 6, 7, 5, 6, 6, 5, 8, 5, 1, 5],
     [2,7,2,7,6,6,7,6,8,8, 6, 2, 7, 8, 7, 5, 8, 4, 7, 6],
     [2,7,2,7,7,7,7,6,7,6, 2, 7, 6, 1, 1, 8, 3, 7, 1, 9],
     [2,2,2,2,7,7,6,6,8,2, 2, 7, 5, 1, 1, 5, 7, 6, 8, 1],
     [2,2,2,2,2,2,2,2,2,2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]],
    [[4,1,1,1,4,4, 9,1, 1, 1,10, 4, 1, 8, 4, 8, 9,10, 5, 3],
     [4,1,4,1,1,4, 4,1, 9, 4, 9, 6,10, 6, 9, 9, 6,10, 6, 1],
     [4,1,1,1,4,4, 1,4,10, 4, 4, 4, 6,10, 8, 9, 9,10, 7, 7],
     [4,4,1,1,1,4, 9,4,10, 4, 1, 9,10, 1,10, 1, 9, 4, 4, 2],
     [4,4,4,1,4,4, 1,1, 9,10, 4, 6, 9, 6, 9, 1, 9,10, 9,10],
     [4,4,4,1,4,1, 4,1, 1,10, 9, 1, 6, 9, 9, 6,10, 6, 8, 6],
     [4,1,4,1,9,9, 1,9,10,10, 9, 4, 1,10, 1, 6,10, 3, 1, 9],
     [4,1,4,1,1,1, 1,9, 1, 9, 4, 1, 9, 8, 8,10, 7, 1, 8, 5],
     [4,4,4,4,1,1, 9,9,10, 4, 4, 1, 6, 8, 8, 6, 1, 9,10, 8],
     [4,4,4,4,4,4, 4,4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]],
    [[ 6, 3, 3, 3, 6, 6,10, 3, 3, 3, 9, 6, 3, 5, 6, 5,10, 9, 7, 2],
     [ 6, 3, 6, 3, 3, 6, 6, 3,10, 6,10, 1, 9, 1,10,10, 1, 9, 1, 3],
     [ 6, 3, 3, 3, 6, 6, 3, 6, 9, 6, 6, 6, 1, 9, 5,10,10, 9, 4, 4],
     [ 6, 6, 3, 3, 3, 6,10, 6, 9, 6, 3,10, 9, 3, 9, 3,10, 6, 6, 8],
     [ 6, 6, 6, 3, 6, 6, 3, 3,10, 9, 6, 1,10, 1,10, 3,10, 9,10, 9],
     [ 6, 6, 6, 3, 6, 3, 6, 3, 3, 9,10, 3, 1,10,10, 1, 9, 1, 5, 1],
     [ 6, 3, 6, 3,10,10, 3,10, 9, 9,10, 6, 3, 9, 3, 1, 9, 2, 3,10],
     [ 6, 3, 6, 3, 3, 3, 3,10, 3,10, 6, 3,10, 5, 5, 9, 4, 3, 5, 7],
     [ 6, 6, 6, 6, 3, 3,10,10, 9, 6, 6, 3, 1, 5, 5, 1, 3,10, 9, 5],
     [ 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]],
    [[ 8, 4, 4, 4, 8, 8, 5, 4, 4, 4, 7, 8, 4,10, 8,10, 5, 7, 2, 1],
     [ 8, 4, 8, 4, 4, 8, 8, 4, 5, 8, 5, 9, 7, 9, 5, 5, 9, 7, 9, 4],
     [ 8, 4, 4, 4, 8, 8, 4, 8, 7, 8, 8, 8, 9, 7,10, 5, 5, 7, 6, 6],
     [ 8, 8, 4, 4, 4, 8, 5, 8, 7, 8, 4, 5, 7, 4, 7, 4, 5, 8, 8, 3],
     [ 8, 8, 8, 4, 8, 8, 4, 4, 5, 7, 8, 9, 5, 9, 5, 4, 5, 7, 5, 7],
     [ 8, 8, 8, 4, 8, 4, 8, 4, 4, 7, 5, 4, 9, 5, 5, 9, 7, 9,10, 9],
     [ 8, 4, 8, 4, 5, 5, 4, 5, 7, 7, 5, 8, 4, 7, 4, 9, 7, 1, 4, 5],
     [ 8, 4, 8, 4, 4, 4, 4, 5, 4, 5, 8, 4, 5,10,10, 7, 6, 4,10, 2],
     [ 8, 8, 8, 8, 4, 4, 5, 5, 7, 8, 8, 4, 9,10,10, 9, 4, 5, 7,10],
     [ 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]],
    [[10, 9, 9, 9,10,10, 8, 9, 9, 9, 5,10, 9, 2,10, 2, 8, 5, 3, 7],
     [10, 9,10, 9, 9,10,10, 9, 8,10, 8, 4, 5, 4, 8, 8, 4, 5, 4, 9],
     [10, 9, 9, 9,10,10, 9,10, 5,10,10,10, 4, 5, 2, 8, 8, 5, 1, 1],
     [10,10, 9, 9, 9,10, 8,10, 5,10, 9, 8, 5, 9, 5, 9, 8,10,10, 6],
     [10,10,10, 9,10,10, 9, 9, 8, 5,10, 4, 8, 4, 8, 9, 8, 5, 8, 5],
     [10,10,10, 9,10, 9,10, 9, 9, 5, 8, 9, 4, 8, 8, 4, 5, 4, 2, 4],
     [10, 9,10, 9, 8, 8, 9, 8, 5, 5, 8,10, 9, 5, 9, 4, 5, 7, 9, 8],
     [10, 9,10, 9, 9, 9, 9, 8, 9, 8,10, 9, 8, 2, 2, 5, 1, 9, 2, 3],
     [10,10,10,10, 9, 9, 8, 8, 5,10,10, 9, 4, 2, 2, 4, 9, 8, 5, 2],
     [10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10]],
], dtype=int)


In [ ]:
# Data loading & entropy
# ================================================================

def load_sequences():
    sequences = []
    for s in range(5):
        for j in range(20):
            sequences.append([NUM_TO_LETTER[int(n)] for n in VAL_DATA[s, :, j]])
    return sequences


def compute_entropy(sequence):
    entropies = []
    for i in range(len(sequence)):
        history = ['blank'] + sequence[:i + 1]
        u, c    = np.unique(history, return_counts=True)
        p       = c / len(history)
        entropies.append(-np.sum(p * np.log2(p + 1e-12)))
    return entropies


In [ ]:
# ================================================================
# Network initialisation  —  entries ~ Normal(W_MEAN/N, W_SD/sqrt(N))
#   => common-mode eigenvalue ~ W_MEAN, random spectral radius ~ W_SD
# ================================================================

def init_W():
    """Draw a Gaussian matrix and normalise it EXACTLY (same recipe as the
    renorm): empirical entry-mean = W_MEAN/N and entry-SD = W_SD/sqrt(N),
    so the common-mode eigenvalue is W_MEAN and the random radius is W_SD
    with no per-draw sampling noise."""
    return renormalize_W_eff(np.random.normal(0.0, 1.0, (N, N)))


def create_stimuli():
    """Each stimulus is a fixed input vector, drawn once per network.
       'sparse_uniform': a distinct random ~SPARSITY*N subset is active,
                         magnitudes ~ U[0,1].
       'dense_uniform' : all N neurons active, magnitudes ~ U[0,1].
    """
    stimuli  = {}
    n_active = max(1, int(round(SPARSITY * N)))
    for name in ALL_STIMULI:
        if INPUT_MODE == 'sparse_uniform':
            I      = np.zeros(N)
            idx    = np.random.choice(N, n_active, replace=False)
            I[idx] = np.random.uniform(0.0, 1.0, n_active)
        elif INPUT_MODE == 'dense_uniform':
            I = np.random.uniform(0.0, 1.0, N)
        else:
            raise ValueError(f'unknown INPUT_MODE: {INPUT_MODE}')
        stimuli[name] = I * stim_strength
    return stimuli


def relu(x):
    return np.maximum(0, x)


def renormalize_W_eff(M):
    """Reset the random part to spectral radius W_SD (entry-SD = W_SD/sqrt(N))
    while pinning the mean to the defined common-mode eigenvalue W_MEAN
    (entry-mean = W_MEAN/N). Subtracting the mean first decouples the SD
    rescaling from the mean, so the mean is held exactly throughout.
    No epsilon, no eigenvalues; the only degenerate case (zero-variance
    matrix) is an early return."""
    mu          = M.mean()
    sd          = M.std()
    target_mean = W_MEAN / N
    if sd == 0:
        return np.full_like(M, target_mean)
    return (M - mu) / sd * (W_SD / np.sqrt(N)) + target_mean


In [ ]:
# ================================================================
# Core sequence runner  —  recurrent Hebbian (on r) + afferent gain (on I_raw)
# ================================================================

def run_sequence(seq, W, stimuli):
    eff       = np.ones((N, N))   # recurrent Hebbian multiplicative state on W
    g         = np.ones(N)        # afferent (feedforward) per-synapse input gain
    responses = []

    rec_h = np.exp(-1.0 / TAU_HEBBIAN)    # recurrent recovery factor
    rec_g = np.exp(-1.0 / TAU_AFFERENT)   # afferent  recovery factor

    for stim_name in seq:
        I_raw = stimuli[stim_name]
        I     = g * I_raw                 # gain-modulated feedforward drive
        r     = np.zeros(N)

        W_eff = W * eff
        if NORMALIZE_EV:
            W_eff = renormalize_W_eff(W_eff)   # pin mean & random radius before the dynamics

        for _ in range(n_iter):
            r += dt * (-r + relu(W_eff @ r + I - theta0)) / tau

        responses.append(r.copy())

        # numeric guards (keep running even if facilitation blows up)
        r_sig = np.nan_to_num(r, nan=0.0, posinf=0.0, neginf=0.0)
        r_sig = np.clip(r_sig, -1e6, 1e6)

        # ---- recurrent: Hebbian facilitation driven by the OUTPUTS r ----
        #      eff is a pure accumulator (grows + decays toward 1); it is never
        #      reconstructed from W, so there is no division by near-zero W.
        if ETA_HEBBIAN != 0:
            denom = N * np.mean(r_sig ** 2)
            if denom > 0:
                eff += ETA_HEBBIAN * np.outer(r_sig, r_sig) / denom
            eff = 1.0 + (eff - 1.0) * rec_h
            eff = np.clip(eff, -1e6, 1e6)

        # ---- afferent: per-synapse gain driven by the RAW input magnitude ----
        if ETA_AFFERENT != 0:
            g  = g + ETA_AFFERENT * I_raw     # facilitate (>0) / depress (<0) by magnitude
            g  = 1.0 + (g - 1.0) * rec_g      # recover toward baseline 1
            g  = np.clip(g, MIN_GAIN, MAX_GAIN)

    return np.array(responses)


In [ ]:
# ================================================================
# # Binning  — computed from all sequences combined
# ================================================================

def bin_data(all_resp, all_ent, all_id):
    emin, emax = all_ent.min(), all_ent.max()
    edges      = np.linspace(emin, emax, N_BINS + 1)
    centers    = (edges[:-1] + edges[1:]) / 2
    bidx       = np.clip(np.digitize(all_ent, edges[:-1]) - 1, 0, N_BINS - 1)

    bin_means, bin_ent_means, bin_stats = {}, {}, {}
    for b in range(N_BINS):
        mask = bidx == b
        if mask.sum() == 0:
            continue
        bin_ent_means[b] = float(np.mean(all_ent[mask]))
        bin_stats[b] = {
            'mean':   float(np.mean(all_ent[mask])),
            'median': float(np.median(all_ent[mask])),
            'center': float(centers[b]),
        }
        bin_means[b] = {}
        for s in ALL_STIMULI:
            smask = mask & (all_id == s)
            bin_means[b][s] = (np.mean(all_resp[smask], axis=0)
                               if smask.sum() > 0 else np.zeros(N))
    return bin_means, bin_ent_means, bin_stats


def compute_corr_matrices(bin_means):
    corr_mats = {}
    for b, bm in bin_means.items():
        mat          = np.array([bm[s] for s in ALL_STIMULI])
        C            = np.corrcoef(mat)
        corr_mats[b] = np.nan_to_num(C, nan=0.0)
    return corr_mats


def average_bin_means(bm_list):
    avg = {}
    for b in range(N_BINS):
        bms = [bml[b] for bml in bm_list if b in bml]
        if not bms:
            continue
        avg[b] = {}
        for s in ALL_STIMULI:
            vecs      = [bm[s] for bm in bms if s in bm]
            avg[b][s] = np.mean(vecs, axis=0) if vecs else np.zeros(N)
    return avg


def average_corr_matrices(cm_list):
    avg = {}
    for b in range(N_BINS):
        mats = [cm[b] for cm in cm_list if b in cm]
        if mats:
            avg[b] = np.mean(mats, axis=0)
    return avg


# ================================================================


In [ ]:
# ================================================================
# # Utilities
# ================================================================

def save_fig(fig, name):
    base = os.path.join(OUTPUT_DIR, name)
    fig.savefig(base + '.png', dpi=150, bbox_inches='tight')
    fig.savefig(base + '.eps', format='eps', bbox_inches='tight')
    plt.close(fig)
    print(f'  saved  {name}.png  +  .eps')


def bin_subtitle(b, bin_stats):
    """Three lines stacked vertically."""
    bs = bin_stats.get(b, {})
    return (f'mean  = {bs.get("mean",   0):.2f}\n'
            f'med   = {bs.get("median", 0):.2f}\n'
            f'ctr   = {bs.get("center", 0):.2f}')


def norm_sizes(vals, s_min=30, s_max=280):
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(len(vals), (s_min + s_max) / 2)
    return s_min + (s_max - s_min) * (vals - vmin) / (vmax - vmin)


# ================================================================
# Plot 1 — Activity heatmaps
# ================================================================

def plot_activity_heatmaps(bin_means, bin_stats, prefix):
    all_vals = np.concatenate([
        [bin_means[b][s] for s in ALL_STIMULI]
        for b in range(N_BINS) if b in bin_means
    ])
    vmin, vmax = float(all_vals.min()), float(all_vals.max())

    fig_h = max(7, N // 15) + 2.0
    fig, axes = plt.subplots(1, N_BINS + 1,
                              figsize=(4.5 * (N_BINS + 1), fig_h),
                              gridspec_kw={'wspace': 0.05})
    im = None
    for b in range(N_BINS):
        ax = axes[b]
        if b not in bin_means:
            ax.set_visible(False)
            continue
        mat = np.array([bin_means[b][s] for s in ALL_STIMULI]).T
        im  = ax.imshow(mat, aspect='auto', cmap='hot_r',
                        vmin=vmin, vmax=vmax, origin='upper',
                        interpolation='nearest')
        ax.set_xticks(range(10))
        ax.set_xticklabels(ALL_STIMULI, fontsize=FONT_SIZE_TICK)
        ax.set_xlabel('Stimulus', fontsize=FONT_SIZE_BASE)
        if b == 0:
            ax.set_ylabel('Neuron', fontsize=FONT_SIZE_BASE)
            ax.tick_params(axis='y', labelsize=FONT_SIZE_TICK)
        else:
            ax.set_yticks([])
        ax.set_title(bin_subtitle(b, bin_stats),
                     fontsize=FONT_SIZE_BASE - 2, pad=8, linespacing=1.5)

    # Extra panel for colorbar
    ax_cb = axes[-1]
    ax_cb.set_visible(False)
    if im is not None:
        cbar = fig.colorbar(im, ax=ax_cb, fraction=0.8, pad=0.04)
        cbar.set_label('Activity', fontsize=FONT_SIZE_CBAR)
        cbar.ax.tick_params(labelsize=FONT_SIZE_TICK)

    fig.suptitle(f'{prefix}  —  Mean Activity per Stimulus per Entropy Bin',
                 fontweight='bold', fontsize=FONT_SIZE_TITLE, y=1.08)
    save_fig(fig, f'{prefix}_activity_heatmaps')


# ================================================================
# Plot 2 — Correlation matrices  (square cells)
# ================================================================

def plot_corr_matrices(corr_mats, bin_stats, prefix):
    cell_size = 0.55
    mat_side  = 10 * cell_size
    fig_h     = mat_side + 4.0
    fig_w     = (N_BINS + 1) * (mat_side + 0.3) + 1.5

    fig, axes = plt.subplots(1, N_BINS + 1,
                              figsize=(fig_w, fig_h),
                              gridspec_kw={'wspace': 0.08})
    im = None
    for b in range(N_BINS):
        ax = axes[b]
        if b not in corr_mats:
            ax.set_visible(False)
            continue
        im = ax.imshow(corr_mats[b], cmap='RdBu_r', vmin=-1, vmax=1,
                       aspect='equal')
        ax.set_xticks(range(10))
        ax.set_xticklabels(ALL_STIMULI, fontsize=FONT_SIZE_TICK - 2)
        ax.set_yticks(range(10))
        ax.set_yticklabels(ALL_STIMULI if b == 0 else [],
                           fontsize=FONT_SIZE_TICK - 2)
        ax.set_xlabel('Stimulus', fontsize=FONT_SIZE_BASE - 2)
        if b == 0:
            ax.set_ylabel('Stimulus', fontsize=FONT_SIZE_BASE - 2)
        ax.set_title(bin_subtitle(b, bin_stats),
                     fontsize=FONT_SIZE_BASE - 2, pad=8, linespacing=1.5)

    # Extra panel for colorbar
    ax_cb = axes[-1]
    ax_cb.set_visible(False)
    if im is not None:
        cbar = fig.colorbar(im, ax=ax_cb, fraction=0.8, pad=0.04)
        cbar.set_label('Pearson r', fontsize=FONT_SIZE_CBAR)
        cbar.ax.tick_params(labelsize=FONT_SIZE_TICK)

    fig.suptitle(f'{prefix}  —  Stimulus Correlation Matrices per Entropy Bin',
                 fontweight='bold', fontsize=FONT_SIZE_TITLE, y=1.10)
    save_fig(fig, f'{prefix}_corr_matrices')


# ================================================================
# Build PCA datasets
# ================================================================

def build_datasets(bin_means, corr_mats, bin_ent_means,
                   all_resp, all_ent, all_id):
    Xb, sb, eb, mb, blb = [], [], [], [], []
    for b in range(N_BINS):
        if b not in corr_mats:
            continue
        C = corr_mats[b]
        for si, s in enumerate(ALL_STIMULI):
            Xb.append(C[si])
            sb.append(si)
            eb.append(bin_ent_means.get(b, 0.0))
            vec = bin_means[b].get(s, np.zeros(N)) if b in bin_means else np.zeros(N)
            mb.append(float(np.mean(vec)))
            blb.append(b)

    ds_b = (np.array(Xb), np.array(sb), np.array(eb),
            np.array(mb), np.array(blb))

    sc   = np.array([STIM_TO_IDX[s] for s in all_id])
    mc   = np.mean(all_resp, axis=1)
    ds_c = (all_resp, sc, all_ent, mc)

    return ds_b, ds_c


# ================================================================
# Plot 3 — PCA rows b and c, 3 colour schemes (6 individual square figs)
# ================================================================

def plot_pca_individual(ds_b, ds_c, prefix):
    col_modes = [
        ('stim',     'By stimulus'),
        ('entropy',  'By entropy'),
        ('activity', 'By mean activity'),
    ]
    col_cmaps = {'entropy': 'viridis', 'activity': 'plasma'}
    col_clabs = {'entropy': 'Entropy (bits)', 'activity': 'Mean activity'}

    row_configs = [
        ('b', ds_b[:4], 'Corr-matrix rows (50 pts: 5 bins × 10 stim)'),
        ('c', ds_c,     f'All trial responses ({ds_c[0].shape[0]} pts)'),
    ]

    for var, (X, s_l, e_l, m_l), var_label in row_configs:
        n_comp = min(2, X.shape[0] - 1, X.shape[1])
        if n_comp < 2:
            continue
        pca    = PCA(n_components=2)
        coords = pca.fit_transform(X)
        ev     = pca.explained_variance_ratio_

        for mode, mode_label in col_modes:
            fig, ax = plt.subplots(figsize=(8, 8))

            if mode == 'stim':
                for si, s in enumerate(ALL_STIMULI):
                    mask = s_l == si
                    ax.scatter(coords[mask, 0], coords[mask, 1],
                               color=STIM_COLORS[si],
                               s=60, alpha=0.85, linewidths=0)
                # Stimulus colorbar
                sm   = plt.cm.ScalarMappable(cmap=STIM_CMAP, norm=STIM_NORM)
                sm.set_array([])
                cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04,
                                    ticks=range(10))
                cbar.ax.set_yticklabels(ALL_STIMULI, fontsize=FONT_SIZE_TICK)
                cbar.set_label('Stimulus', fontsize=FONT_SIZE_CBAR)
            else:
                vals = e_l if mode == 'entropy' else m_l
                sc_p = ax.scatter(coords[:, 0], coords[:, 1], c=vals,
                                  cmap=col_cmaps[mode],
                                  s=60, alpha=0.85, linewidths=0)
                cbar = fig.colorbar(sc_p, ax=ax, fraction=0.046, pad=0.04)
                cbar.set_label(col_clabs[mode], fontsize=FONT_SIZE_CBAR)
                cbar.ax.tick_params(labelsize=FONT_SIZE_TICK)

            ax.set_box_aspect(1)
            ax.set_xlabel(f'PC1 ({ev[0]:.1%})', fontsize=FONT_SIZE_BASE)
            ax.set_ylabel(f'PC2 ({ev[1]:.1%})', fontsize=FONT_SIZE_BASE)
            ax.tick_params(labelsize=FONT_SIZE_TICK)
            ax.set_title(f'{var_label}\n{mode_label}',
                         fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=10)
            plt.tight_layout()
            save_fig(fig, f'{prefix}_pca_{var}_{mode}')


# ================================================================
# Plot 4 — Trajectory PCA (5 square panels)
# ================================================================

def plot_pca_trajectory(ds_b, bin_stats, prefix):
    X, sb, eb, mb, blb = ds_b

    n_comp = min(2, X.shape[0] - 1, X.shape[1])
    if n_comp < 2:
        print('  [skip] pca_trajectory: not enough data')
        return

    pca    = PCA(n_components=2)
    coords = pca.fit_transform(X)
    ev     = pca.explained_variance_ratio_

    available_bins = sorted(set(blb))
    coords_by_bin  = {}
    for b in available_bins:
        coords_by_bin[b] = {}
        for si in range(10):
            mask = (blb == b) & (sb == si)
            if mask.sum() > 0:
                coords_by_bin[b][si] = coords[mask][0]

    n_avail     = len(available_bins)
    point_sizes = np.linspace(40, 200, n_avail)

    pad  = 0.12 * max(np.ptp(coords[:, 0]), np.ptp(coords[:, 1]))
    xlim = (coords[:, 0].min() - pad, coords[:, 0].max() + pad)
    ylim = (coords[:, 1].min() - pad, coords[:, 1].max() + pad)

    panel_size = 5.0
    n_cols     = n_avail + 1     # extra column for colorbar
    fig, axes  = plt.subplots(1, n_cols,
                               figsize=(panel_size * n_cols, panel_size + 2.5),
                               gridspec_kw={'wspace': 0.15})

    for panel_idx, b in enumerate(available_bins):
        ax = axes[panel_idx]

        for si in range(10):
            if si not in coords_by_bin.get(b, {}):
                continue

            if panel_idx > 0:
                n_seg       = panel_idx
                grey_levels = np.linspace(0.75, 0.2, n_seg)
                for seg_i in range(n_seg):
                    prev_b = available_bins[seg_i]
                    next_b = available_bins[seg_i + 1]
                    if (si not in coords_by_bin.get(prev_b, {}) or
                            si not in coords_by_bin.get(next_b, {})):
                        continue
                    x0, y0 = coords_by_bin[prev_b][si]
                    x1, y1 = coords_by_bin[next_b][si]
                    g = grey_levels[seg_i]
                    ax.plot([x0, x1], [y0, y1], '-',
                            color=(g, g, g), linewidth=1.5,
                            zorder=2, solid_capstyle='round')

            x, y = coords_by_bin[b][si]
            ax.scatter(x, y, color=STIM_COLORS[si],
                       s=point_sizes[panel_idx], zorder=3,
                       linewidths=0.5, edgecolors='white')

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_box_aspect(1)
        ax.set_xlabel(f'PC1 ({ev[0]:.1%})', fontsize=FONT_SIZE_BASE - 2)
        if panel_idx == 0:
            ax.set_ylabel(f'PC2 ({ev[1]:.1%})', fontsize=FONT_SIZE_BASE - 2)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=FONT_SIZE_TICK - 2)
        ax.set_title(bin_subtitle(b, bin_stats),
                     fontsize=FONT_SIZE_BASE - 2, pad=8, linespacing=1.5)

    # 6th panel: colorbar only
    ax_cb = axes[-1]
    ax_cb.set_visible(False)
    sm    = plt.cm.ScalarMappable(cmap=STIM_CMAP, norm=STIM_NORM)
    sm.set_array([])
    cbar  = fig.colorbar(sm, ax=ax_cb, fraction=0.8, pad=0.04,
                         ticks=range(10))
    cbar.ax.set_yticklabels(ALL_STIMULI, fontsize=FONT_SIZE_TICK - 2)
    cbar.set_label('Stimulus', fontsize=FONT_SIZE_CBAR)

    fig.suptitle(f'{prefix}  —  PCA Trajectory (corr-matrix rows)',
                 fontweight='bold', fontsize=FONT_SIZE_TITLE, y=1.10)
    save_fig(fig, f'{prefix}_pca_trajectory')


# ================================================================
# Plot 5 — PCA with entropy & activity axes overlay (2 square panels)
# ================================================================

def plot_pca_axes(ds_b, prefix):
    X, sb, eb, mb, blb = ds_b

    n_comp = min(2, X.shape[0] - 1, X.shape[1])
    if n_comp < 2:
        print('  [skip] pca_axes: not enough data')
        return

    pca    = PCA(n_components=2)
    coords = pca.fit_transform(X)
    ev     = pca.explained_variance_ratio_
    comps  = pca.components_

    ridge = Ridge(alpha=1.0)
    ridge.fit(X, eb);  w_ent_pc = comps @ ridge.coef_
    ridge.fit(X, mb);  w_act_pc = comps @ ridge.coef_

    data_range  = max(np.ptp(coords[:, 0]), np.ptp(coords[:, 1]))
    arrow_scale = data_range * 0.35
    centroid    = coords.mean(axis=0)

    def line_endpoints(w_pc):
        norm = np.linalg.norm(w_pc)
        if norm < 1e-12:
            return None
        w_hat = w_pc / norm
        return (centroid - arrow_scale * w_hat,
                centroid + arrow_scale * w_hat)

    panel_configs = [
        ('activity', norm_sizes(mb), 'Point size ∝ mean activity'),
        ('entropy',  norm_sizes(eb), 'Point size ∝ entropy'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(24, 8),
                              gridspec_kw={'wspace': 0.25})

    for ax, (size_key, sizes, panel_title) in zip(axes[:2], panel_configs):
        for si in range(10):
            mask = sb == si
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       color=STIM_COLORS[si], s=sizes[mask],
                       alpha=0.85, linewidths=0.5, edgecolors='white',
                       zorder=3)

        ep = line_endpoints(w_ent_pc)
        if ep is not None:
            ax.plot([ep[0][0], ep[1][0]], [ep[0][1], ep[1][1]],
                    '-', color='steelblue', linewidth=2.0,
                    label='entropy axis', zorder=5)

        ap = line_endpoints(w_act_pc)
        if ap is not None:
            ax.plot([ap[0][0], ap[1][0]], [ap[0][1], ap[1][1]],
                    '-', color='tomato', linewidth=2.0,
                    label='activity axis', zorder=5)

        ax.set_box_aspect(1)
        ax.set_xlabel(f'PC1 ({ev[0]:.1%})', fontsize=FONT_SIZE_BASE)
        ax.set_ylabel(f'PC2 ({ev[1]:.1%})', fontsize=FONT_SIZE_BASE)
        ax.tick_params(labelsize=FONT_SIZE_TICK)
        ax.set_title(panel_title, fontsize=FONT_SIZE_TITLE,
                     fontweight='bold', pad=10)
        # Legend only for the axis lines
        ax.legend(fontsize=FONT_SIZE_TICK - 2, framealpha=0.6, loc='best')

    # 3rd panel: stimulus colorbar only
    ax_cb = axes[2]
    ax_cb.set_visible(False)
    sm    = plt.cm.ScalarMappable(cmap=STIM_CMAP, norm=STIM_NORM)
    sm.set_array([])
    cbar  = fig.colorbar(sm, ax=ax_cb, fraction=0.8, pad=0.04,
                         ticks=range(10))
    cbar.ax.set_yticklabels(ALL_STIMULI, fontsize=FONT_SIZE_TICK)
    cbar.set_label('Stimulus', fontsize=FONT_SIZE_CBAR)

    fig.suptitle(f'{prefix}  —  PCA with Entropy and Activity Axes',
                 fontweight='bold', fontsize=FONT_SIZE_TITLE, y=1.04)
    save_fig(fig, f'{prefix}_pca_axes')


# ================================================================
# Driver — called for each network and the summary
# ================================================================

def make_all_plots(bin_means, corr_mats, bin_ent_means, bin_stats,
                   all_resp, all_ent, all_id, prefix):
    ds_b, ds_c = build_datasets(bin_means, corr_mats, bin_ent_means,
                                 all_resp, all_ent, all_id)
    plot_activity_heatmaps(bin_means, bin_stats, prefix)
    plot_corr_matrices(corr_mats, bin_stats, prefix)
    plot_pca_individual(ds_b, ds_c, prefix)
    plot_pca_trajectory(ds_b, bin_stats, prefix)
    plot_pca_axes(ds_b, prefix)


In [ ]:
# ================================================================
# Sequence selection — boring vs max-entropy permutations
# ================================================================
PERMUTATIONS = {
    'boring':     {'col':  0, 'label': 'Boring sequence (constant stimulus)'},
    'maxentropy': {'col': 19, 'label': 'Max-entropy sequence (all stimuli distinct)'},
}


def get_variation_sequences(col):
    """Return the 5 variation sequences (as letters) for a given VAL_DATA column."""
    return [[NUM_TO_LETTER[int(n)] for n in VAL_DATA[s, :, col]] for s in range(5)]


def plot_permutation(W, stimuli, col, perm_label, prefix, tag):
    seqs  = get_variation_sequences(col)                  # 5 sequences, each length 10
    resps = [run_sequence(seq, W, stimuli) for seq in seqs]  # each -> (n_steps, N)

    L     = len(seqs[0])
    steps = np.arange(1, L + 1)

    # Shared activity colour scale across the 5 heatmaps
    all_act    = np.concatenate([r.ravel() for r in resps])
    vmin, vmax = float(all_act.min()), float(all_act.max())

    # Per-step mean / SD across neurons, shared y-scale across the 5 line plots
    means = [r.mean(axis=1) for r in resps]
    sds   = [r.std(axis=1)  for r in resps]
    y_hi  = max(float((m + s).max()) for m, s in zip(means, sds))
    y_lo  = min(float((m - s).min()) for m, s in zip(means, sds))
    span  = y_hi - y_lo if y_hi > y_lo else 1.0
    pad   = 0.06 * span

    fig, axes = plt.subplots(
        2, N_BINS if False else 5,
        figsize=(5 * 4.3, 9.8),
        gridspec_kw={'height_ratios': [3.0, 1.5], 'hspace': 0.30, 'wspace': 0.10},
    )

    im = None
    for v in range(5):
        seq = seqs[v]
        r   = resps[v]                                    # (L, N)

        # ---- top: activity heatmap (neurons x steps) ----
        ax_h = axes[0, v]
        im = ax_h.imshow(
            r.T, aspect='auto', cmap='hot_r', vmin=vmin, vmax=vmax,
            origin='upper', interpolation='nearest',
            extent=[0.5, L + 0.5, N - 0.5, -0.5],
        )
        ax_h.set_xticks(steps)
        ax_h.set_xticklabels(seq, fontsize=FONT_SIZE_TICK - 4)
        ax_h.set_title(f'Variation {v + 1}',
                       fontsize=FONT_SIZE_BASE, pad=8)
        if v == 0:
            ax_h.set_ylabel('Neuron', fontsize=FONT_SIZE_BASE - 2)
            ax_h.tick_params(axis='y', labelsize=FONT_SIZE_TICK - 2)
        else:
            ax_h.set_yticks([])

        # ---- bottom: mean activity across neurons +/- SD ----
        ax_l = axes[1, v]
        m, s = means[v], sds[v]
        ax_l.fill_between(steps, m - s, m + s,
                          color='tab:blue', alpha=0.25, linewidth=0)
        ax_l.plot(steps, m, '-o', color='tab:blue',
                  linewidth=2.0, markersize=5, zorder=3)
        ax_l.set_xlim(0.5, L + 0.5)
        ax_l.set_ylim(y_lo - pad, y_hi + pad)
        ax_l.set_xticks(steps)
        ax_l.set_xticklabels(seq, fontsize=FONT_SIZE_TICK - 4)
        ax_l.set_xlabel('Stimulus (step)', fontsize=FONT_SIZE_BASE - 4)
        ax_l.grid(True, alpha=0.25)
        if v == 0:
            ax_l.set_ylabel('Mean activity\n(\u00b1 SD over neurons)',
                            fontsize=FONT_SIZE_BASE - 5)
            ax_l.tick_params(axis='y', labelsize=FONT_SIZE_TICK - 3)
        else:
            ax_l.set_yticklabels([])

    # Shared colorbar for the heatmap row
    if im is not None:
        cbar = fig.colorbar(im, ax=list(axes[0, :]),
                            fraction=0.018, pad=0.012)
        cbar.set_label('Activity', fontsize=FONT_SIZE_CBAR - 2)
        cbar.ax.tick_params(labelsize=FONT_SIZE_TICK - 3)

    ent_final = compute_entropy(seqs[0])[-1]
    fig.suptitle(f'{prefix}  \u2014  {perm_label}   '
                 f'(final entropy \u2248 {ent_final:.2f} bits)',
                 fontweight='bold', fontsize=FONT_SIZE_TITLE, y=0.98)
    save_fig(fig, f'{prefix}_{tag}')


In [ ]:
# ================================================================
# Driver
# ================================================================

def make_all_plots(bin_means, corr_mats, bin_ent_means, bin_stats,
                   all_resp, all_ent, all_id, prefix):
    ds_b, ds_c = build_datasets(bin_means, corr_mats, bin_ent_means,
                                 all_resp, all_ent, all_id)
    plot_activity_heatmaps(bin_means, bin_stats, prefix)
    plot_corr_matrices(corr_mats, bin_stats, prefix)
    plot_pca_individual(ds_b, ds_c, prefix)
    plot_pca_trajectory(ds_b, bin_stats, prefix)
    plot_pca_axes(ds_b, prefix)


# ================================================================
# Main
# ================================================================
if __name__ == '__main__':
    print('=' * 60)
    print('Neural Adaptation — Afferent-Gain Example Plots')
    print(f'  W_MEAN={W_MEAN}  W_SD={W_SD}')
    print(f'  ETA_HEBBIAN(rec,+)={ETA_HEBBIAN}  ETA_AFFERENT(ff,-)={ETA_AFFERENT}')
    print(f'  TAU_HEBBIAN={TAU_HEBBIAN}  TAU_AFFERENT={TAU_AFFERENT}'
          f'  NORMALIZE_EV={NORMALIZE_EV}')
    print(f'  N_NETWORKS={N_NETWORKS}')
    print(f'  Output: {OUTPUT_DIR}')
    print('=' * 60)

    bm_list, cm_list             = [], []
    resp_list, ent_list, id_list = [], [], []
    shared_bem = None
    shared_bs  = None
    seqs       = load_sequences()

    for net_idx in range(N_NETWORKS):
        print(f'\n[Network {net_idx + 1}/{N_NETWORKS}]')
        W       = init_W()
        stimuli = create_stimuli()

        all_resp, all_ent, all_id = [], [], []
        for seq in seqs:
            ents  = compute_entropy(seq)
            resps = run_sequence(seq, W, stimuli)
            all_resp.extend(resps)
            all_ent.extend(ents)
            all_id.extend(seq)

        all_resp = np.array(all_resp)
        all_ent  = np.array(all_ent)
        all_id   = np.array(all_id)

        bm, bem, bs = bin_data(all_resp, all_ent, all_id)
        cm          = compute_corr_matrices(bm)

        bm_list.append(bm)
        cm_list.append(cm)
        resp_list.append(all_resp)
        ent_list.append(all_ent)
        id_list.append(all_id)

        if shared_bem is None:
            shared_bem = bem
            shared_bs  = bs

        print('  Saving per-network standard plots …')
        make_all_plots(bm, cm, bem, bs,
                       all_resp, all_ent, all_id,
                       prefix=f'net{net_idx + 1}')

        print('  Saving boring / max-entropy mean-activity panels …')
        for tag, info in PERMUTATIONS.items():
            plot_permutation(W, stimuli, info['col'], info['label'],
                             prefix=f'net{net_idx + 1}', tag=tag)

    # ---- Summary (standard suite, averaged across networks) ----
    print('\n[Summary — averaged across networks]')
    avg_bm      = average_bin_means(bm_list)
    avg_cm      = average_corr_matrices(cm_list)
    pooled_resp = np.vstack(resp_list)
    pooled_ent  = np.concatenate(ent_list)
    pooled_id   = np.concatenate(id_list)

    make_all_plots(avg_bm, avg_cm, shared_bem, shared_bs,
                   pooled_resp, pooled_ent, pooled_id,
                   prefix='netavg')

    print(f'\nAll done!  Figures saved to: {os.path.abspath(OUTPUT_DIR)}')
